# Part 1: Standard KB with a Search Index knowledge source

In this notebook you will wire up a **pre-existing search index** as a
**`searchIndex` knowledge source**, create a **knowledge base**, and run a
grounded **retrieve** call — all using the `2026-05-01-preview` REST API.

> **No SDK required** — every call is a plain `requests` HTTP call so you can see
> exactly what goes over the wire.

### Prerequisites

| Item | Details |
|------|---------|
| Restored indexes | Run [restore.ipynb](restore.ipynb) first to create the `hrdocs` and `healthdocs` indexes. |
| `.env` file | In the `notebooks/` directory. Contains `BIGBUGS_ENDPOINT` and `BIGBUGS_API_KEY`. |
| Python packages | `requests`, `python-dotenv` (already in `requirements.txt`). |

## 1 — Setup: load environment and create a session

In [ ]:
import json, os, time, uuid
from pathlib import Path
from datetime import datetime, timezone

import requests

# ── Load secrets from .env in the notebook directory ─────────────────
ENV_PATH = Path(".").resolve() / ".env"

def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        values[k.strip()] = v.strip()
    return values

env = load_env(ENV_PATH)
ENDPOINT = env["BIGBUGS_ENDPOINT"].rstrip("/")
API_KEY  = env["BIGBUGS_API_KEY"]
API_VERSION = "2026-05-01-preview"

session = requests.Session()
session.headers.update({"api-key": API_KEY, "Accept": "application/json"})

def url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{ENDPOINT}{path}{sep}api-version={API_VERSION}"

def pp(response: requests.Response):
    print(f"{response.status_code} {response.reason}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text[:500])

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")

# Pre-existing indexes restored by restore.ipynb
HRDOCS_INDEX      = "hrdocs"
HEALTHDOCS_INDEX  = "healthdocs"

KS_NAME  = f"lab532-searchindex-{stamp}"
KB_NAME  = f"lab532-searchkb-{stamp}"

print(f"Endpoint          : {ENDPOINT}")
print(f"hrdocs index      : {HRDOCS_INDEX}")
print(f"healthdocs index  : {HEALTHDOCS_INDEX}")
print(f"KS                : {KS_NAME}")
print(f"KB                : {KB_NAME}")

## 2 — Quick service health check

In [ ]:
r = session.get(url("/servicestats"))
pp(r)

## 3 — Verify restored indexes

Confirm the `hrdocs` and `healthdocs` indexes exist and contain documents.
If these are empty, run [restore.ipynb](restore.ipynb) first.

In [ ]:
for idx in [HRDOCS_INDEX, HEALTHDOCS_INDEX]:
    r = session.get(url(f"/indexes('{idx}')/docs/$count"))
    print(f"{idx}: {r.text} documents (HTTP {r.status_code})")

## 4 — Create a `searchIndex` knowledge source

The knowledge source points at the pre-existing `hrdocs` index.
Fields must match the index schema restored by `restore.ipynb`.

In [ ]:
ks_body = {
    "name": KS_NAME,
    "description": "Lab 532 searchIndex knowledge source (hrdocs)",
    "kind": "searchIndex",
    "searchIndexParameters": {
        "searchIndexName": HRDOCS_INDEX,
        "sourceDataFields": [
            {"name": "uid"}, {"name": "snippet_parent_id"},
            {"name": "blob_path"}, {"name": "snippet"}
        ],
        "searchFields": [{"name": "snippet"}],
        "semanticConfigurationName": "semantic-configuration",
    },
}

r = session.put(url(f"/knowledgesources('{KS_NAME}')"), json=ks_body,
                headers={"Prefer": "return=representation"})
pp(r)

## 5 — Create a knowledge base referencing the source

In [ ]:
kb_body = {
    "name": KB_NAME,
    "description": "Lab 532 knowledge base — searchIndex backed",
    "outputMode": "extractiveData",
    "retrievalReasoningEffort": {"kind": "minimal"},
    "knowledgeSources": [{"name": KS_NAME}],
}

r = session.put(url(f"/knowledgebases('{KB_NAME}')"), json=kb_body,
                headers={"Prefer": "return=representation"})
pp(r)

## 6 — Retrieve: ask the KB a question

Key parameters:
- `maxOutputDocuments` controls how many grounding references come back.
- `includeActivity: true` shows which sources were consulted.
- `knowledgeSourceParams` lets you pass per-source overrides at query time.

In [ ]:
retrieve_body = {
    "intents": [{"type": "semantic", "search": "Tell me about Zava and its history"}],
    "retrievalReasoningEffort": {"kind": "minimal"},
    "includeActivity": True,
    "maxOutputDocuments": 3,
    "knowledgeSourceParams": [
        {
            "knowledgeSourceName": KS_NAME,
            "kind": "searchIndex",
            "includeReferences": True,
            "includeReferenceSourceData": True,
            "alwaysQuerySource": True,
        }
    ],
}

r = session.post(url(f"/knowledgebases('{KB_NAME}')/retrieve"), json=retrieve_body)
pp(r)

### Try a second query

In [ ]:
retrieve_body_2 = {
    "intents": [{"type": "semantic", "search": "What employee perks and benefits are available?"}],
    "retrievalReasoningEffort": {"kind": "minimal"},
    "includeActivity": True,
    "maxOutputDocuments": 3,
    "knowledgeSourceParams": [
        {
            "knowledgeSourceName": KS_NAME,
            "kind": "searchIndex",
            "includeReferences": True,
            "includeReferenceSourceData": True,
            "alwaysQuerySource": True,
        }
    ],
}

r = session.post(url(f"/knowledgebases('{KB_NAME}')/retrieve"), json=retrieve_body_2)
pp(r)

## 7 — Cleanup

Delete the KB and KS. The `hrdocs` and `healthdocs` indexes are **not** deleted —
they are shared across all lab notebooks.

In [ ]:
for label, path in [
    ("KB", f"/knowledgebases('{KB_NAME}')"),
    ("KS", f"/knowledgesources('{KS_NAME}')"),
]:
    r = session.delete(url(path))
    print(f"Delete {label}: {r.status_code} {r.reason}")

## Known limitations on the current BigBugs stamp

| Feature | Status |
|---------|--------|
| `searchIndex` retrieve | ✅ Works |
| `maxOutputDocuments` (request-level) | ✅ Works — controls final output count |
| Per-source `maxOutputDocuments` | ⚠️ Accepted but no observable effect yet |
| KB-stored per-source retrieve defaults | ❌ Rejected at KB create time |
| `top` parameter on retrieve | ❌ Rejected — use `maxOutputDocuments` |
| `file` KS retrieve | ❌ Not supported on this stamp |

➡️ Continue to [Part 2: Add Fabric IQ as a first-class knowledge source type](part2-fabric-iq-to-kb.ipynb).